# DL_01: Data Preparation & Preprocessing Pipeline
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook establishes reproducible data preparation and preprocessing pipelines for both Phase 2 benchmark streams:
1. **Corr2Cause**: Abstract causal inference formulated as Natural Language Inference (NLI) premise-hypothesis entailment pairs.
2. **BioCause**: Candidate-conditioned directed causal relation classification using annotated argument spans from open-access PMC biomedical articles.

Key requirements:
- Prepare Corr2Cause and BioCause datasets independently (strict boundary enforcement).
- Document dataset schemas, label distributions, and directional label semantics.
- Ingest canonical input files from `data/raw/` and `data/processed/`.
- Preserve the canonical Corr2Cause split counts: Train (205,734), Validation (1,076), and Test (1,162).
- Maintain the 12-train / 4-validation / 4-test PMC article-level split isolation for BioCause to prevent cross-article data leakage.
- Construct explicit argument boundary markers (`[E1]...[/E1]` and `[E2]...[/E2]`) preserving original argument order and contextual text.
- Define expected model input tensors and target outputs for transformer fine-tuning in `DL_02` and `DL_03`.


## 2. Experimental Methodology & Theoretical Background

### Causal Relation Classification vs. Causal Entailment
- **Corr2Cause (Symbolic NLI)**: Evaluates whether language models deduce valid causal graph relations solely from statistical independence statements:
  - **Premise**: Verbalized statistical independence constraints among variables.
  - **Hypothesis**: Directed causal conjecture (e.g. "*A causes B.*")
  - **Target Label**: `1` (Entailed) or `0` (Refuted).
  - **Canonical Split Distribution**: Train = 205,734; Validation = 1,076; Test = 1,162. Validation must remain strictly separate from training.

- **BioCause (Biomedical Candidate-Conditioned Relation Classification)**: Given co-occurring biomedical entity spans ($Arg_1$, $Arg_2$) in scientific text, classify whether the directed relationship $Arg_1 \to Arg_2$ holds:
  - **Label Semantics**: Label `1` denotes that $Arg_1$ causes $Arg_2$. Label `0` denotes that $Arg_1$ does not represent the gold directed relation $Arg_1 \to Arg_2$ under the candidate-generation convention (which includes reversed directional pairs $(Arg_2, Arg_1)$ as well as non-causal pairs). Label `0` must not be mischaracterized as universally non-causal.
  - **Class Imbalance**: Candidate pairing produces 10,086 training pairs (621 positive, 9,465 negative), an actual negative-to-positive ratio of **15.24:1** (Validation: 131 pos / 1,783 neg = 13.61:1; Test: 100 pos / 1,308 neg = 13.08:1).

### Article-Level Split Isolation (Zero Data Leakage)
BioCause strictly partitions the 20 PMC publications by article ID:
- **Train Split**: 12 PMC Articles (621 gold causal relations, 10,086 candidate pairs)
- **Validation Split**: 4 PMC Articles (131 gold causal relations, 1,914 candidate pairs)
- **Test Split**: 4 PMC Articles (100 gold causal relations, 1,408 candidate pairs)


## 3. Expected Inputs

| Stream | File Path | Canonical Size / Dimensions | Key Fields |
| :--- | :--- | :--- | :--- |
| **Corr2Cause Train** | `data/raw/train.csv` | 205,734 rows | `input, label, num_variables, template` |
| **Corr2Cause Val** | `data/raw/validation.csv` | 1,076 rows | `input, label, num_variables, template` |
| **Corr2Cause Test** | `data/raw/test.csv` | 1,162 rows | `input, label, num_variables, template` |
| **BioCause Candidates** | `data/processed/biocause_candidates.csv` | 13,408 candidate pairs | `candidate_id, article_id, section_id, arg1_text, arg2_text, between_text, label, split...` |
| **BioCause Gold Relations** | `data/processed/biocause_relations.csv` | 852 gold relations | `relation_id, article_id, cause_id, effect_id...` |


## 4. Technical Responsibilities & Implementation Requirements

1. **Environment & Portable Root Discovery**: Locate repo root dynamically and load `dl_utils`.
2. **Corr2Cause Split Ingestion**: Verify exact row counts (205,734 train, 1,076 val, 1,162 test) and verify binary label balance.
3. **BioCause Article-Level Isolation Audit**: Check that article IDs across splits are strictly disjoint.
4. **Argument Marker Construction**: Format BioCause inputs with special tokens `[E1]`, `[/E1]`, `[E2]`, `[/E2]`. Ensure markers reference original text offsets, preserve original text order, include surrounding context, and check for span validity/overlap.
5. **Dynamic Class Weight Calculation**: Compute $w_{\text{pos}} = N_{\text{neg}} / N_{\text{pos}}$ strictly on the training split ($9,465 / 621 \approx 15.24$). Never leak validation or test counts into weight computation.
6. **Token Length Profiling**: Profile sequence lengths under `distilbert-base-uncased` and `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` tokenizers.
7. **Dataset Serialization**: Export clean model-ready splits to `data/processed/`.


---

## Section A — Environment Setup
Set up repository imports, configure reproducibility, detect compute device, and display environment information.

In [1]:
# Section A: Environment Setup & Portable Imports
import sys
import os
import warnings
from pathlib import Path

# ---- Repository root discovery ----
def find_repo_root():
    """Locate repository root across local, Colab, and Kaggle."""
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
import pandas as pd
import numpy as np

print(f'Repository root: [repo_root]/ ({ROOT_DIR.name})')
print(f'Python version:  {sys.version.split()[0]}')
print(f'pandas version:  {pd.__version__}')
print(f'numpy version:   {np.__version__}')

Repository root: [repo_root]/ (causal-nlp-reasoning)
Python version:  3.14.2
pandas version:  3.0.1
numpy version:   2.4.2


In [2]:
# Reproducibility & device detection
dl_utils.set_seed(42)
DEVICE = dl_utils.get_device()
print(f'Compute device:  {DEVICE}')

try:
    import torch
    print(f'PyTorch version: {torch.__version__}')
    if torch.cuda.is_available():
        print(f'CUDA device:     {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch: not installed')

try:
    import transformers
    print(f'Transformers:    {transformers.__version__}')
except ImportError:
    print('Transformers: not installed')

Compute device:  mps
PyTorch version: 2.14.1


Transformers:    4.57.6


---

## Section B — Corr2Cause Data Preparation
Load canonical Corr2Cause CSV splits and verify row counts, columns, labels, missing values, and duplicates.

In [3]:
# Section B: Corr2Cause Data Ingestion & Split Verification

CORR2CAUSE_RAW_DIR = ROOT_DIR / 'data' / 'raw'

c2c_files = {
    'train': CORR2CAUSE_RAW_DIR / 'train.csv',
    'validation': CORR2CAUSE_RAW_DIR / 'validation.csv',
    'test': CORR2CAUSE_RAW_DIR / 'test.csv',
}

# Check that all files exist before proceeding
missing_c2c = [name for name, path in c2c_files.items() if not path.exists()]
if missing_c2c:
    msg = (
        f'MISSING Corr2Cause raw files: {missing_c2c}\n'
        f'Expected location: data/raw/\n'
        f'Required files: train.csv, validation.csv, test.csv\n'
        f'Download from HuggingFace: https://huggingface.co/datasets/causalnlp/corr2cause\n'
        f'Or run 00_setup.ipynb to automatically stage the datasets.'
    )
    raise FileNotFoundError(msg)

In [4]:
# Load Corr2Cause splits
df_c2c_train = pd.read_csv(c2c_files['train'])
df_c2c_val = pd.read_csv(c2c_files['validation'])
df_c2c_test = pd.read_csv(c2c_files['test'])

print('=== Corr2Cause Split Sizes ===')
print(f'  Train:      {len(df_c2c_train):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"]:,})')
print(f'  Validation: {len(df_c2c_val):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"]:,})')
print(f'  Test:       {len(df_c2c_test):>10,} (expected {dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"]:,})')

# Verify exact row counts
assert len(df_c2c_train) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['train'], \
    f'Train count mismatch: {len(df_c2c_train)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"]}'
assert len(df_c2c_val) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['validation'], \
    f'Val count mismatch: {len(df_c2c_val)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"]}'
assert len(df_c2c_test) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['test'], \
    f'Test count mismatch: {len(df_c2c_test)} vs {dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"]}'
print('\n✓ All Corr2Cause split counts verified.')

=== Corr2Cause Split Sizes ===
  Train:         205,734 (expected 205,734)
  Validation:      1,076 (expected 1,076)
  Test:            1,162 (expected 1,162)

✓ All Corr2Cause split counts verified.


In [5]:
# Validate required columns
C2C_REQUIRED_COLS = ['input', 'label']
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    dl_utils.validate_dataset_schema(df, C2C_REQUIRED_COLS, f'Corr2Cause-{split_name}')
print('✓ All required Corr2Cause columns present.')

# Verify label format (binary: 0 or 1)
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    unique_labels = set(df['label'].dropna().unique())
    assert unique_labels.issubset({0, 1}), f'{split_name}: non-binary labels found: {unique_labels}'
print('✓ All Corr2Cause labels are binary (0/1).')

# Check for missing values in critical fields
for split_name, df in [('train', df_c2c_train), ('val', df_c2c_val), ('test', df_c2c_test)]:
    null_input = df['input'].isna().sum()
    null_label = df['label'].isna().sum()
    assert null_input == 0, f'{split_name}: {null_input} null inputs'
    assert null_label == 0, f'{split_name}: {null_label} null labels'
print('✓ No missing values in input or label columns.')

# Label distribution
print('\n=== Corr2Cause Label Distributions ===')
for split_name, df in [('train', df_c2c_train), ('validation', df_c2c_val), ('test', df_c2c_test)]:
    vc = df['label'].value_counts().sort_index()
    print(f'  {split_name:>12}: label_0={vc.get(0, 0):>7,}, label_1={vc.get(1, 0):>7,}')

✓ All required Corr2Cause columns present.
✓ All Corr2Cause labels are binary (0/1).
✓ No missing values in input or label columns.

=== Corr2Cause Label Distributions ===
         train: label_0=167,686, label_1= 38,048
    validation: label_0=    937, label_1=    139
          test: label_0=    982, label_1=    180


---

## Section C — BioCause Data Preparation
Load BioCause candidate pairs, verify article-level split isolation, candidate identifiers,
argument text/offsets, binary labels, and expected counts.

In [6]:
# Section C: BioCause Data Ingestion & Article-Level Split Isolation Audit

BIOCAUSE_CANDS_PATH = ROOT_DIR / 'data' / 'processed' / 'biocause_candidates.csv'

if not BIOCAUSE_CANDS_PATH.exists():
    msg = (
        f'MISSING BioCause candidates file: {BIOCAUSE_CANDS_PATH}\n'
        f'This file is generated by 00_setup.ipynb and AML/AML_01_data_features.ipynb.\n'
        f'Run those notebooks first to produce the candidate pairs.'
    )
    raise FileNotFoundError(msg)

df_bio = pd.read_csv(BIOCAUSE_CANDS_PATH, keep_default_na=False)
print(f'BioCause candidates loaded: {len(df_bio):,} rows')
print(f'Columns: {list(df_bio.columns)}')

BioCause candidates loaded: 13,408 rows
Columns: ['candidate_id', 'dataset', 'article_id', 'section_id', 'arg1_id', 'arg1_text', 'arg1_start', 'arg1_end', 'arg2_id', 'arg2_text', 'arg2_start', 'arg2_end', 'char_distance', 'arg1_before_arg2', 'between_text', 'label', 'split']


In [7]:
# Validate BioCause candidates
bio_summary = dl_utils.validate_biocause_candidates(df_bio, 'BioCause Candidates')

print('\n=== BioCause Split Summary ===')
print(f'{"Split":<12} {"Articles":>10} {"Candidates":>12} {"Positive":>10} {"Negative":>10}')
print('-' * 58)
for split_name in ['train', 'validation', 'test']:
    s = bio_summary[split_name]
    print(f'{split_name:<12} {s["articles"]:>10} {s["total"]:>12,} {s["positive"]:>10,} {s["negative"]:>10,}')

print('\n✓ BioCause candidate validation passed (article isolation, labels, IDs).')


=== BioCause Split Summary ===
Split          Articles   Candidates   Positive   Negative
----------------------------------------------------------
train                12       10,086        621      9,465
validation            4        1,914        131      1,783
test                  4        1,408        100      1,308

✓ BioCause candidate validation passed (article isolation, labels, IDs).


In [8]:
# Verify article-level split isolation explicitly
train_articles = set(df_bio[df_bio['split'] == 'train']['article_id'].unique())
val_articles = set(df_bio[df_bio['split'] == 'validation']['article_id'].unique())
test_articles = set(df_bio[df_bio['split'] == 'test']['article_id'].unique())

assert len(train_articles & val_articles) == 0, 'Train/Val article overlap detected!'
assert len(train_articles & test_articles) == 0, 'Train/Test article overlap detected!'
assert len(val_articles & test_articles) == 0, 'Val/Test article overlap detected!'

print(f'Train articles ({len(train_articles)}):  {sorted(train_articles)}')
print(f'Val articles ({len(val_articles)}):    {sorted(val_articles)}')
print(f'Test articles ({len(test_articles)}):   {sorted(test_articles)}')
print('\n✓ Zero cross-split article leakage confirmed.')

Train articles (12):  ['PMC1804205', 'PMC1874608', 'PMC1913099', 'PMC1974823', 'PMC2266911', 'PMC2565068', 'PMC2639726', 'PMC2682197', 'PMC2774163', 'PMC2816692', 'PMC2829055', 'PMC2858072']
Val articles (4):    ['PMC2242835', 'PMC2581603', 'PMC2651894', 'PMC2885601']
Test articles (4):   ['PMC2358977', 'PMC2430206', 'PMC2593050', 'PMC2714965']

✓ Zero cross-split article leakage confirmed.


In [9]:
# Verify expected split counts against canonical values
expected = dl_utils.BIOCAUSE_CANDIDATE_COUNTS

for split_name in ['train', 'validation', 'test']:
    actual = bio_summary[split_name]
    exp = expected[split_name]
    assert actual['total'] == exp['total'], \
        f'{split_name} total: {actual["total"]} vs expected {exp["total"]}'
    assert actual['positive'] == exp['positive'], \
        f'{split_name} positive: {actual["positive"]} vs expected {exp["positive"]}'
    assert actual['negative'] == exp['negative'], \
        f'{split_name} negative: {actual["negative"]} vs expected {exp["negative"]}'

print('✓ All BioCause candidate counts match expected values.')

✓ All BioCause candidate counts match expected values.


In [10]:
# Check for duplicate candidate IDs
dup_count = df_bio['candidate_id'].duplicated().sum()
assert dup_count == 0, f'Found {dup_count} duplicate candidate_ids'
print(f'✓ No duplicate candidate_ids ({len(df_bio):,} unique candidates).')

# Verify argument identifiers and text are present
for col in ['arg1_id', 'arg2_id', 'arg1_text', 'arg2_text']:
    null_count = df_bio[col].isna().sum()
    print(f'  {col}: {null_count} nulls')
    assert null_count == 0, f'{col} has {null_count} null values'

✓ No duplicate candidate_ids (13,408 unique candidates).
  arg1_id: 0 nulls
  arg2_id: 0 nulls
  arg1_text: 0 nulls
  arg2_text: 0 nulls


---

## Section D — Entity Marker Construction
Insert `[E1]...[/E1]` and `[E2]...[/E2]` entity boundary markers using character offsets.
Verify that Arg1 and Arg2 identities are maintained and demonstrate handling of different argument orders.

In [11]:
# Section D: Entity Marker Construction via Source Section Reconstruction
# Check required candidate columns
required_cand_cols = ["arg1_start", "arg1_end", "arg2_start", "arg2_end", "arg1_text", "arg2_text", "section_id"]
missing_cand_cols = [c for c in required_cand_cols if c not in df_bio.columns]
if missing_cand_cols:
    raise ValueError(f"BioCause candidates missing required offset/span columns: {missing_cand_cols}")

# Locate and cache raw section .txt files from data/raw/biocause/BioCause_corpus/ (or fallback data/raw/biocause/)
unique_sec_ids = df_bio["section_id"].unique()
print(f"Loading raw section files for {len(unique_sec_ids)} unique BioCause sections...")

sec_text_cache = {}
missing_sec_files = []

for sec_id in unique_sec_ids:
    try:
        sec_text = dl_utils.load_biocause_section_text(sec_id)
        sec_text_cache[sec_id] = sec_text
    except FileNotFoundError:
        missing_sec_files.append(sec_id)

if missing_sec_files:
    raise FileNotFoundError(
        f"Missing {len(missing_sec_files)} BioCause raw section .txt files in "
        f"data/raw/biocause/BioCause_corpus/ or data/raw/biocause/. "
        f"Sample missing sections: {missing_sec_files[:5]}. "
        f"Run 00_setup.ipynb or ensure the BioCause corpus is staged."
    )

print(f"✓ Successfully loaded {len(sec_text_cache)} BioCause section text files.")

Loading raw section files for 185 unique BioCause sections...
✓ Successfully loaded 185 BioCause section text files.


In [12]:
# Demonstrate marker insertion and span validation on sample candidates
print("=== Entity Marker Construction Examples ===")
sample_rows = df_bio.head(5)

for idx, row in sample_rows.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    marked = dl_utils.reconstruct_biocause_context_with_markers(
        section_text=sec_text,
        arg1_start=int(row["arg1_start"]),
        arg1_end=int(row["arg1_end"]),
        arg2_start=int(row["arg2_start"]),
        arg2_end=int(row["arg2_end"]),
        arg1_text=str(row["arg1_text"]),
        arg2_text=str(row["arg2_text"]),
        candidate_id=str(row["candidate_id"]),
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
    )
    
    # Assert markers are present
    assert "[E1]" in marked and "[/E1]" in marked, "Missing E1 markers"
    assert "[E2]" in marked and "[/E2]" in marked, "Missing E2 markers"
    
    print(f"\n[{idx}] candidate_id={row['candidate_id']}")
    print(f"  Arg1: {row['arg1_text']!r} [{row['arg1_start']}:{row['arg1_end']}]")
    print(f"  Arg2: {row['arg2_text']!r} [{row['arg2_start']}:{row['arg2_end']}]")
    print(f"  Marked & Cropped: {marked[:200]}..." if len(marked) > 200 else f"  Marked: {marked}")

=== Entity Marker Construction Examples ===

[0] candidate_id=PMC1804205-00-TIAB_T27_T29
  Arg1: 'Growth in mild acid (pH 5.8) promotes transcription of genes activated by the response regulator PmrA, but the signalling pathway(s) that mediates this response has thus far remained unexplored' [405:598]
  Arg2: "that this activation requires both PmrA's cognate sensor kinase PmrB, which had been previously shown to respond to Fe3+ and Al3+, and PmrA's post-translational activator PmrD" [615:790]
  Marked & Cropped: Acid pH activation of the PmrA/PmrB two-component regulatory system of Salmonella enterica 
Acid pH often triggers changes in gene expression.
However, little is known about the identity of the gene p...

[1] candidate_id=PMC1804205-00-TIAB_T27_T30
  Arg1: 'Growth in mild acid (pH 5.8) promotes transcription of genes activated by the response regulator PmrA, but the signalling pathway(s) that mediates this response has thus far remained unexplored' [405:598]
  Arg2: 'Substitut

In [13]:
# Reconstruct marked source context for all BioCause candidates
# Every candidate undergoes rigorous source-section reconstruction with entity boundary markers.
# Failed reconstructions (e.g. overlapping spans, out-of-bounds offsets) do NOT receive synthetic fallback text;
# they are flagged with reconstruction_valid = False and excluded from model-ready training splits.
print("Reconstructing marked source context for all BioCause candidates...")
reconstruction_valid = []
marked_texts = []
diagnostics = []

for idx, row in df_bio.iterrows():
    sec_text = sec_text_cache[row["section_id"]]
    try:
        marked = dl_utils.reconstruct_biocause_context_with_markers(
            section_text=sec_text,
            arg1_start=int(row["arg1_start"]),
            arg1_end=int(row["arg1_end"]),
            arg2_start=int(row["arg2_start"]),
            arg2_end=int(row["arg2_end"]),
            arg1_text=str(row["arg1_text"]),
            arg2_text=str(row["arg2_text"]),
            candidate_id=str(row["candidate_id"]),
            max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
        )
        reconstruction_valid.append(True)
        marked_texts.append(marked)
        diagnostics.append("")
    except Exception as e:
        # Invalid candidate: do NOT fabricate fallback context or synthesize text
        reconstruction_valid.append(False)
        marked_texts.append(None)
        diagnostics.append(str(e))

df_bio["reconstruction_valid"] = reconstruction_valid
df_bio["marked_text"] = marked_texts
df_bio["diagnostic"] = diagnostics

n_total = len(df_bio)
n_valid = int(df_bio["reconstruction_valid"].sum())
n_invalid = n_total - n_valid

print(f"\n=== BioCause Source Reconstruction Audit ===")
print(f"Total candidates:                 {n_total:,}")
print(f"Valid source reconstructions:     {n_valid:,} ({n_valid / n_total * 100:.2f}%)")
print(f"Invalid source reconstructions:   {n_invalid:,} ({n_invalid / n_total * 100:.2f}%)")

# Detailed breakdown by split
print(f"\n{'Split':<12} {'Total Cands':>12} {'Model-Eligible':>15} {'Invalid':>10}")
print("-" * 52)
for s in ["train", "validation", "test"]:
    sub = df_bio[df_bio["split"] == s]
    v = int(sub["reconstruction_valid"].sum())
    inv = len(sub) - v
    print(f"{s:<12} {len(sub):>12,} {v:>15,} {inv:>10,}")

# Breakdown of invalid reasons
df_invalid = df_bio[~df_bio["reconstruction_valid"]]
if len(df_invalid) > 0:
    print(f"\n=== Invalid Reconstruction Reasons (Total: {len(df_invalid)}) ===")
    error_categories = []
    for d in df_invalid["diagnostic"]:
        if "Overlapping" in d:
            error_categories.append("Overlapping argument spans")
        elif "Cropping failed" in d:
            error_categories.append("Cropping failed to preserve spans within token budget")
        elif "Invalid arg" in d and "offset" in d:
            error_categories.append("Offset out of section bounds in standoff")
        elif "mismatch" in d:
            error_categories.append("Text span mismatch with source")
        else:
            error_categories.append("Other offset/span error")
    df_invalid_cat = pd.Series(error_categories)
    for cat, count in df_invalid_cat.value_counts().items():
        print(f"  - {cat}: {count} candidates")

# Explicitly filter model-eligible candidates (strictly excluding invalid reconstructions)
df_bio_eligible = df_bio[df_bio["reconstruction_valid"]].copy()

# Detailed assertions on model-eligible candidates
assert (df_bio_eligible["reconstruction_valid"] == True).all(), "Non-valid reconstruction in model-eligible candidates!"
assert df_bio_eligible["marked_text"].notna().all(), "Null marked_text found in model-eligible candidates!"
assert df_bio_eligible["marked_text"].str.contains(r"\[E1\]").all(), "Missing [E1] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[/E1\]").all(), "Missing [/E1] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[E2\]").all(), "Missing [E2] marker!"
assert df_bio_eligible["marked_text"].str.contains(r"\[/E2\]").all(), "Missing [/E2] marker!"
assert df_bio_eligible["candidate_id"].duplicated().sum() == 0, "Duplicate candidate_ids in model-eligible set!"

# Article-level split isolation on model-eligible subset
train_arts = set(df_bio_eligible[df_bio_eligible["split"] == "train"]["article_id"].unique())
val_arts = set(df_bio_eligible[df_bio_eligible["split"] == "validation"]["article_id"].unique())
test_arts = set(df_bio_eligible[df_bio_eligible["split"] == "test"]["article_id"].unique())
assert len(train_arts & val_arts) == 0, "Train/Val article leakage in eligible set!"
assert len(train_arts & test_arts) == 0, "Train/Test article leakage in eligible set!"
assert len(val_arts & test_arts) == 0, "Val/Test article leakage in eligible set!"

print(f"\n✓ Verified {len(df_bio_eligible):,} model-eligible candidates across all 3 splits.")
print("✓ All assertions passed: verified entity spans, 4 boundary markers, 0 cross-split leakage.")

Reconstructing marked source context for all BioCause candidates...



=== BioCause Source Reconstruction Audit ===
Total candidates:                 13,408
Valid source reconstructions:     13,186 (98.34%)
Invalid source reconstructions:   222 (1.66%)

Split         Total Cands  Model-Eligible    Invalid
----------------------------------------------------
train              10,086           9,908        178
validation          1,914           1,884         30
test                1,408           1,394         14

=== Invalid Reconstruction Reasons (Total: 222) ===
  - Overlapping argument spans: 188 candidates
  - Cropping failed to preserve spans within token budget: 22 candidates
  - Offset out of section bounds in standoff: 12 candidates

✓ Verified 13,186 model-eligible candidates across all 3 splits.
✓ All assertions passed: verified entity spans, 4 boundary markers, 0 cross-split leakage.


In [14]:
# Demonstrate different argument orders
print("=== Argument Order Verification (Model-Eligible Candidates) ===")
offset_cols = ["arg1_start", "arg1_end", "arg2_start", "arg2_end"]
missing_offsets = [c for c in offset_cols if c not in df_bio_eligible.columns]
if missing_offsets:
    raise ValueError(
        f"BioCause candidates missing required offset columns: {missing_offsets}. "
        f"Offsets are strictly required for the entity marker workflow."
    )

# Find cases where arg1 appears after arg2 in text
arg2_first = df_bio_eligible[df_bio_eligible["arg2_start"] < df_bio_eligible["arg1_start"]]
arg1_first = df_bio_eligible[df_bio_eligible["arg1_start"] < df_bio_eligible["arg2_start"]]

print(f"  Arg1 before Arg2: {len(arg1_first):,} candidates")
print(f"  Arg2 before Arg1: {len(arg2_first):,} candidates")

# Show one example of each ordering
if len(arg1_first) > 0:
    row = arg1_first.iloc[0]
    print(f"\nArg1-first example (candidate {row['candidate_id']}):")
    print(f"  Arg1=[{row['arg1_start']}:{row['arg1_end']}] \"{row['arg1_text']}\"")
    print(f"  Arg2=[{row['arg2_start']}:{row['arg2_end']}] \"{row['arg2_text']}\"")
    print(f"  Marked: {row['marked_text'][:200]}")

if len(arg2_first) > 0:
    row = arg2_first.iloc[0]
    print(f"\nArg2-first example (candidate {row['candidate_id']}):")
    print(f"  Arg1=[{row['arg1_start']}:{row['arg1_end']}] \"{row['arg1_text']}\"")
    print(f"  Arg2=[{row['arg2_start']}:{row['arg2_end']}] \"{row['arg2_text']}\"")
    print(f"  Marked: {row['marked_text'][:200]}")

=== Argument Order Verification (Model-Eligible Candidates) ===
  Arg1 before Arg2: 6,593 candidates
  Arg2 before Arg1: 6,593 candidates

Arg1-first example (candidate PMC1804205-00-TIAB_T27_T29):
  Arg1=[405:598] "Growth in mild acid (pH 5.8) promotes transcription of genes activated by the response regulator PmrA, but the signalling pathway(s) that mediates this response has thus far remained unexplored"
  Arg2=[615:790] "that this activation requires both PmrA's cognate sensor kinase PmrB, which had been previously shown to respond to Fe3+ and Al3+, and PmrA's post-translational activator PmrD"
  Marked: Acid pH activation of the PmrA/PmrB two-component regulatory system of Salmonella enterica 
Acid pH often triggers changes in gene expression.
However, little is known about the identity of the gene p

Arg2-first example (candidate PMC1804205-00-TIAB_T29_T27):
  Arg1=[615:790] "that this activation requires both PmrA's cognate sensor kinase PmrB, which had been previously shown to 

---

## Section E — Tokenizer Profiling
Load tokenizers, register entity markers as special tokens, profile token lengths,
calculate truncation statistics, and apply context cropping where necessary.

In [15]:
# Section E: Tokenizer Profiling (DistilBERT & PubMedBERT)
# Profile sequence lengths across both datasets to document sensible sequence length limits.

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
    print("WARNING: transformers not installed. Skipping tokenizer profiling.")
    print("Install with: pip install -r requirements-dl.txt")

In [16]:
if HAS_TRANSFORMERS:
    # 1. Profile DistilBERT on Corr2Cause inputs
    print("=== 1. Corr2Cause Tokenizer Profiling (DistilBERT) ===")
    distilbert_tokenizer = AutoTokenizer.from_pretrained(dl_utils.DISTILBERT_CHECKPOINT)
    
    # Format sample Corr2Cause inputs
    sample_c2c_texts = [
        f"[CLS] {row['input']} [SEP]"
        for _, row in df_c2c_train.head(1000).iterrows()
    ]
    
    c2c_profile = dl_utils.profile_token_lengths(
        texts=sample_c2c_texts,
        tokenizer=distilbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH,  # 256 tokens
    )
    
    print(f"  Samples: {c2c_profile['count']:,}")
    print(f"  Min:     {c2c_profile['min']} tokens")
    print(f"  Mean:    {c2c_profile['mean']} tokens")
    print(f"  Median:  {c2c_profile['median']:.0f} tokens")
    print(f"  P95:     {c2c_profile['p95']:.0f} tokens")
    print(f"  P99:     {c2c_profile['p99']:.0f} tokens")
    print(f"  Max:     {c2c_profile['max']} tokens")
    print(f"  Within {dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH} budget: {c2c_profile['pct_within_budget']:.1f}%")
    print(f"\n✓ Documented setting for Corr2Cause: MAX_LENGTH = {dl_utils.DEFAULT_CORR2CAUSE_MAX_LENGTH} tokens.")

=== 1. Corr2Cause Tokenizer Profiling (DistilBERT) ===


Token indices sequence length is longer than the specified maximum sequence length for this model (702 > 512). Running this sequence through the model will result in indexing errors


  Samples: 1,000
  Min:     86 tokens
  Mean:    206.92 tokens
  Median:  172 tokens
  P95:     385 tokens
  P99:     705 tokens
  Max:     710 tokens
  Within 256 budget: 84.8%

✓ Documented setting for Corr2Cause: MAX_LENGTH = 256 tokens.


In [17]:
if HAS_TRANSFORMERS:
    # 2. Profile PubMedBERT on marked BioCause inputs (model-eligible candidates)
    print("=== 2. BioCause Tokenizer Profiling (PubMedBERT) ===")
    pubmedbert_tokenizer = dl_utils.prepare_tokenizer_and_inputs(
        checkpoint_name=dl_utils.PUBMEDBERT_CHECKPOINT,
        max_length=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,
        additional_special_tokens=dl_utils.ENTITY_MARKERS,
    )
    print(f"  Vocab size with markers: {len(pubmedbert_tokenizer)}")
    
    # Verify entity markers are recognized tokens
    for marker in dl_utils.ENTITY_MARKERS:
        token_id = pubmedbert_tokenizer.convert_tokens_to_ids(marker)
        assert token_id != pubmedbert_tokenizer.unk_token_id, f"{marker} not recognized!"
    
    bio_profile = dl_utils.profile_token_lengths(
        texts=df_bio_eligible["marked_text"].head(1000).tolist(),
        tokenizer=pubmedbert_tokenizer,
        max_samples=1000,
        target_budget=dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH,  # 384 tokens
    )
    
    print(f"  Samples: {bio_profile['count']:,}")
    print(f"  Min:     {bio_profile['min']} tokens")
    print(f"  Mean:    {bio_profile['mean']} tokens")
    print(f"  Median:  {bio_profile['median']:.0f} tokens")
    print(f"  P95:     {bio_profile['p95']:.0f} tokens")
    print(f"  P99:     {bio_profile['p99']:.0f} tokens")
    print(f"  Max:     {bio_profile['max']} tokens")
    print(f"  Within {dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH} budget: {bio_profile['pct_within_budget']:.1f}%")
    print(f"\n✓ Documented setting for BioCause: MAX_LENGTH = {dl_utils.DEFAULT_BIOCAUSE_MAX_LENGTH} tokens.")

=== 2. BioCause Tokenizer Profiling (PubMedBERT) ===


Token indices sequence length is longer than the specified maximum sequence length for this model (405 > 384). Running this sequence through the model will result in indexing errors


  Vocab size with markers: 30526


  Samples: 1,000
  Min:     173 tokens
  Mean:    383.38 tokens
  Median:  403 tokens
  P95:     418 tokens
  P99:     426 tokens
  Max:     428 tokens
  Within 384 budget: 30.0%

✓ Documented setting for BioCause: MAX_LENGTH = 384 tokens.


---

## Section F — Class Imbalance
Calculate the BioCause positive class weight using the training split only.

In [18]:
# Section F: Dynamic Training Class Weight Calculation
# Calculated strictly from model-eligible training candidates

df_bio_train_eligible = df_bio_eligible[df_bio_eligible['split'] == 'train']
n_pos_train = int((df_bio_train_eligible['label'] == 1).sum())
n_neg_train = int((df_bio_train_eligible['label'] == 0).sum())

pos_weight = dl_utils.calculate_dynamic_class_weight(
    n_negative=n_neg_train,
    n_positive=n_pos_train,
)

print('=== BioCause Dynamic Class Weight ===')
print(f'  Model-eligible training positive: {n_pos_train:,}')
print(f'  Model-eligible training negative: {n_neg_train:,}')
print(f'  pos_weight = {n_neg_train} / {n_pos_train} = {pos_weight:.4f}')

n_pos_val = int((df_bio_eligible[(df_bio_eligible['split'] == 'validation') & (df_bio_eligible['label'] == 1)]['label']).count())
n_pos_test = int((df_bio_eligible[(df_bio_eligible['split'] == 'test') & (df_bio_eligible['label'] == 1)]['label']).count())

print(f'\n✓ Class weight computed strictly from model-eligible training split.')
print(f'  NOTE: Validation ({n_pos_val} pos) and test ({n_pos_test} pos) '
      f'labels NOT used in weight calculation.')

=== BioCause Dynamic Class Weight ===
  Model-eligible training positive: 620
  Model-eligible training negative: 9,288
  pos_weight = 9288 / 620 = 14.9806

✓ Class weight computed strictly from model-eligible training split.
  NOTE: Validation (131 pos) and test (100 pos) labels NOT used in weight calculation.


---

## Section G — Export Processed Datasets
Export six model-ready CSV files to `data/processed/`.

In [19]:
# Section G: Export Model-Ready Datasets

PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Corr2Cause exports ---
# Add split column for downstream clarity
df_c2c_train_out = df_c2c_train.copy()
df_c2c_train_out['split'] = 'train'
df_c2c_val_out = df_c2c_val.copy()
df_c2c_val_out['split'] = 'validation'
df_c2c_test_out = df_c2c_test.copy()
df_c2c_test_out['split'] = 'test'

c2c_exports = {
    'dl_corr2cause_train.csv': df_c2c_train_out,
    'dl_corr2cause_val.csv': df_c2c_val_out,
    'dl_corr2cause_test.csv': df_c2c_test_out,
}

print('=== Exporting Corr2Cause Processed Datasets ===')
for fname, df in c2c_exports.items():
    path = str(PROCESSED_DIR / fname)
    dl_utils.export_predictions(df, path, format_type='csv')
    print(f'  ✓ data/processed/{fname}: {len(df):>10,} rows')

=== Exporting Corr2Cause Processed Datasets ===


  ✓ data/processed/dl_corr2cause_train.csv:    205,734 rows
  ✓ data/processed/dl_corr2cause_val.csv:      1,076 rows
  ✓ data/processed/dl_corr2cause_test.csv:      1,162 rows


In [20]:
# --- BioCause exports ---
# Select columns needed by downstream DL notebooks (strictly model-eligible source-verified candidates)
biocause_export_cols = [
    col for col in [
        'candidate_id', 'article_id', 'section_id',
        'arg1_id', 'arg2_id', 'arg1_text', 'arg2_text',
        'arg1_start', 'arg1_end', 'arg2_start', 'arg2_end',
        'between_text', 'section_text', 'marked_text',
        'label', 'split', 'arg1_before_arg2',
    ] if col in df_bio_eligible.columns
]

bio_exports = {
    'dl_biocause_train.csv': df_bio_eligible[df_bio_eligible['split'] == 'train'][biocause_export_cols],
    'dl_biocause_val.csv': df_bio_eligible[df_bio_eligible['split'] == 'validation'][biocause_export_cols],
    'dl_biocause_test.csv': df_bio_eligible[df_bio_eligible['split'] == 'test'][biocause_export_cols],
}

print('\n=== Exporting BioCause Model-Eligible Processed Datasets ===')
for fname, df in bio_exports.items():
    path = str(PROCESSED_DIR / fname)
    dl_utils.export_predictions(df, path, format_type='csv')
    print(f'  ✓ data/processed/{fname}: {len(df):>10,} rows')

# --- Export BioCause Reconstruction Diagnostics ---
# Separate diagnostics CSV containing all invalid candidates with reconstruction failure reasons
diag_cols = [
    col for col in [
        'candidate_id', 'article_id', 'section_id',
        'arg1_id', 'arg2_id', 'arg1_text', 'arg2_text',
        'arg1_start', 'arg1_end', 'arg2_start', 'arg2_end',
        'label', 'split', 'diagnostic',
    ] if col in df_bio.columns
]
df_diagnostics_export = df_bio[~df_bio['reconstruction_valid']][diag_cols]
diag_path = str(PROCESSED_DIR / 'dl_biocause_reconstruction_diagnostics.csv')
dl_utils.export_predictions(df_diagnostics_export, diag_path, format_type='csv')
print(f'  ✓ data/processed/dl_biocause_reconstruction_diagnostics.csv: {len(df_diagnostics_export):>10,} rows')


=== Exporting BioCause Model-Eligible Processed Datasets ===


  ✓ data/processed/dl_biocause_train.csv:      9,908 rows
  ✓ data/processed/dl_biocause_val.csv:      1,884 rows
  ✓ data/processed/dl_biocause_test.csv:      1,394 rows
  ✓ data/processed/dl_biocause_reconstruction_diagnostics.csv:        222 rows


In [21]:
# Final verification: check all 6 model-ready export files and diagnostics file exist, read back, and verify schemas and row counts
expected_exports = {
    "dl_corr2cause_train.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["train"], ["input", "label", "split"]),
    "dl_corr2cause_val.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["validation"], ["input", "label", "split"]),
    "dl_corr2cause_test.csv": (dl_utils.CORR2CAUSE_SPLIT_COUNTS["test"], ["input", "label", "split"]),
    "dl_biocause_train.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["train"]["total"], ["candidate_id", "marked_text", "label", "split"]),
    "dl_biocause_val.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["validation"]["total"], ["candidate_id", "marked_text", "label", "split"]),
    "dl_biocause_test.csv": (dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS["test"]["total"], ["candidate_id", "marked_text", "label", "split"]),
}

print("\n=== Export & Read-Back Verification ===")
all_ok = True
for fname, (exp_rows, req_cols) in expected_exports.items():
    fpath = PROCESSED_DIR / fname
    rel_path = f"data/processed/{fname}"
    if not fpath.exists():
        print(f"  ✗ {rel_path} MISSING")
        all_ok = False
        continue

    size_mb = fpath.stat().st_size / (1024 * 1024)
    # Read back and verify schema and row count
    df_readback = pd.read_csv(fpath)
    actual_rows = len(df_readback)
    assert actual_rows == exp_rows, f"{fname} row mismatch: {actual_rows} vs expected {exp_rows}"
    dl_utils.validate_dataset_schema(df_readback, req_cols, dataset_name=fname)
    # Assert no null marked_text values exist in BioCause splits
    if "marked_text" in df_readback.columns:
        assert df_readback["marked_text"].notna().all(), f"{fname} contains null marked_text!"
    print(f"  ✓ {rel_path}: {actual_rows:>10,} rows ({size_mb:.2f} MB) — schema & row counts verified")

# Verify diagnostics export
diag_fpath = PROCESSED_DIR / "dl_biocause_reconstruction_diagnostics.csv"
if diag_fpath.exists():
    df_diag_readback = pd.read_csv(diag_fpath)
    assert len(df_diag_readback) == len(df_diagnostics_export), (
        f"Diagnostics row mismatch: {len(df_diag_readback)} vs expected {len(df_diagnostics_export)}"
    )
    diag_size_mb = diag_fpath.stat().st_size / (1024 * 1024)
    print(f"  ✓ data/processed/dl_biocause_reconstruction_diagnostics.csv: {len(df_diag_readback):>10,} rows ({diag_size_mb:.2f} MB) — diagnostics verified")
else:
    print("  ✗ data/processed/dl_biocause_reconstruction_diagnostics.csv MISSING")
    all_ok = False

if all_ok:
    print("\n✓ All 6 processed datasets and diagnostics exported and verified successfully via read-back.")
else:
    raise RuntimeError("Export verification failed: some files missing or invalid.")


=== Export & Read-Back Verification ===


  ✓ data/processed/dl_corr2cause_train.csv:    205,734 rows (366.84 MB) — schema & row counts verified
  ✓ data/processed/dl_corr2cause_val.csv:      1,076 rows (1.40 MB) — schema & row counts verified
  ✓ data/processed/dl_corr2cause_test.csv:      1,162 rows (1.13 MB) — schema & row counts verified
  ✓ data/processed/dl_biocause_train.csv:      9,908 rows (27.41 MB) — schema & row counts verified


  ✓ data/processed/dl_biocause_val.csv:      1,884 rows (4.85 MB) — schema & row counts verified
  ✓ data/processed/dl_biocause_test.csv:      1,394 rows (3.74 MB) — schema & row counts verified
  ✓ data/processed/dl_biocause_reconstruction_diagnostics.csv:        222 rows (0.12 MB) — diagnostics verified

✓ All 6 processed datasets and diagnostics exported and verified successfully via read-back.


## 5. Expected Outputs & Evaluation Criteria

- `data/processed/dl_corr2cause_{train,val,test}.csv`: Cleaned NLI splits conforming to canonical split counts.
- `data/processed/dl_biocause_{train,val,test}.csv`: Marked relation candidate datasets with preserved 12/4/4 article grouping.
- **Evaluation Criteria**:
  - 100% split integrity: Zero overlap between article IDs in train, val, and test.
  - Argument marker completeness: Both entity spans intact post-formatting and tokenization.
  - Class weight verification: Dynamic training ratio $w_{\text{pos}} \approx 15.24$ computed strictly from training data.
